# 58. Automation & Scheduling — `schedule · pyautogui`
**Python Libraries Lab · Phase 3 · Files & Automation**

_Run jobs on a timetable, and control the mouse and keyboard._

⚙️ **Runs in Colab** after the setup cell below.

Automation means letting the computer do repetitive work for you. The schedule library runs Python functions on a timetable, like every Monday at nine, or every ten minutes, using plain English style code. PyAutoGUI goes further and controls the mouse and keyboard directly, so it can operate programs that have no API, the way a person would.

**In this notebook:** In Colab you will schedule ShopEasy jobs, a stock check every few seconds, a daily report at a fixed time, and a job that cancels itself, then watch them run. For your laptop you will get a safe PyAutoGUI script that moves the mouse, types into Notepad or TextEdit, and takes a screenshot.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
**schedule** runs fully in Colab. **PyAutoGUI** controls a real mouse and keyboard, so it only works on your own computer: the last steps write a safe practice script for your laptop.

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q schedule
import importlib.metadata as md
for pkg in ['schedule']:
    print(pkg, md.version(pkg))

## Step 1 · Your first scheduled jobs
Register jobs with plain-English timing, then let `run_pending` run whatever is due.

In [ ]:
import schedule, time
from datetime import datetime

log = []
def check_stock():
    log.append(("stock", datetime.now().strftime("%H:%M:%S")))
    print(f"{log[-1][1]}  checked stock levels")

def sales_report():
    log.append(("report", datetime.now().strftime("%H:%M:%S")))
    print(f"{log[-1][1]}  sent sales report")

schedule.clear()
schedule.every(2).seconds.do(check_stock)
schedule.every(5).seconds.do(sales_report)
schedule.every().day.at("09:00").do(sales_report).tag("daily")     # real-life timing
schedule.every().monday.at("08:30").do(sales_report).tag("weekly")
for job in schedule.get_jobs():
    print(job)

## Step 2 · Run the scheduler for 11 seconds
A real script would loop forever with `while True`.

In [ ]:
end = time.time() + 11
while time.time() < end:
    schedule.run_pending()
    time.sleep(0.2)
print("\nRuns:", [kind for kind, _ in log])
print("Next daily report at:", schedule.next_run("daily") if hasattr(schedule, "next_run") else [j.next_run for j in schedule.get_jobs("daily")])

## Step 3 · Jobs that stop themselves, and tags
Return `schedule.CancelJob` to run once; cancel groups of jobs by tag.

In [ ]:
def flash_sale_reminder():
    print("Flash sale starts now!")
    return schedule.CancelJob

schedule.clear()
schedule.every(1).seconds.do(flash_sale_reminder)
schedule.every(1).seconds.do(lambda: print("ping")).tag("pings")
for _ in range(3):
    schedule.run_pending(); time.sleep(1.05)
schedule.clear("pings")
print("Jobs left:", schedule.get_jobs())

## Step 4 · PyAutoGUI practice script (for your laptop)
Moves the mouse in a square, types a note and takes a screenshot. Fail-safe: slam the mouse into a screen corner to stop.

In [ ]:
%%writefile autogui_demo.py
import platform, subprocess, time
import pyautogui

pyautogui.FAILSAFE = True          # move the mouse to a screen corner to abort
pyautogui.PAUSE = 0.4              # small pause after every action

w, h = pyautogui.size()
print(f"Screen {w}x{h}. Starting in 3 seconds... (move mouse to a corner to stop)")
time.sleep(3)

x, y = w // 2, h // 2
for dx, dy in [(200, 0), (0, 200), (-200, 0), (0, -200)]:
    pyautogui.moveTo(x + dx, y + dy, duration=0.4)
    x, y = x + dx, y + dy

if platform.system() == "Windows":
    subprocess.Popen(["notepad.exe"])
elif platform.system() == "Darwin":
    subprocess.Popen(["open", "-a", "TextEdit"])
time.sleep(2)
pyautogui.write("Hello from ShopEasy automation!", interval=0.05)
pyautogui.press("enter")
pyautogui.screenshot("screen.png")
print("Done. Saved screen.png")

## Step 5 · Check the script (safe in Colab)
Colab has no screen, so we only check the script is valid Python. Run it on your laptop.

In [ ]:
import py_compile
py_compile.compile("autogui_demo.py", doraise=True)
print("autogui_demo.py is valid. On your laptop: pip install pyautogui, then python autogui_demo.py")

## ✍️ Your turn
Clear all jobs, schedule a job that appends `'tick'` to a list `ticks` every 1 second, run the scheduler for about 3.5 seconds, and store `len(ticks)` in `answer` (should be 3).

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 3, f"Expected 3, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
schedule.clear()
ticks = []
schedule.every(1).seconds.do(lambda: ticks.append("tick"))
end = time.time() + 3.5
while time.time() < end:
    schedule.run_pending(); time.sleep(0.1)
answer = len(ticks)

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Job**
- **every().day.at()**
- **run_pending**
- **Mouse / keyboard control**
- **Fail-safe**
- **Screenshot**

**Next:** 59. Tkinter Desktop Apps (`tkinter`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/59_tkinter.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install schedule pyautogui
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.
4. **Run the app:** download `autogui_demo.py` (Colab: Files panel on the left → right-click → Download) into the `python-lab` folder, then run `python autogui_demo.py` in the same terminal.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.